# When the old task is well learned: does the bracket work on harder data?

The decider test. Same tests as the MNIST and CIFAR notebooks, on two datasets where task A can be learned properly:
- **`fashion`**: Fashion-MNIST (clothing photos), the same MLP on raw pixels
- **`cifar_feat`**: CIFAR-10 photos passed through a **frozen** ResNet-18 trained on ImageNet (the "ready-made eye"); only a small MLP on top learns

**How to switch:** change `DATASET` in Cell 1, then rerun Cells 1 to 5. Run `cifar_feat` first (it's the main test).

Kernel: `E:\wordstest\.venv\Scripts\python.exe`. Press **Ctrl+S** after each long cell.

## Cell 1: settings
**Should print:** the dataset and the folders.

In [7]:
import os
BASE = r"F:\forgetting"
DATASET = "cifar_feat"        # "cifar_feat" (main test) or "fashion"
DATA_DIR = os.path.join(BASE, "data")
RESULTS_DIR = os.path.join(BASE, "results_" + DATASET)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["TORCH_HOME"] = os.path.join(BASE, "torch_cache")   # pretrained weights go on F, not the full disk
DS = DATASET
SEEDS = 5
EPOCHS = 2 if DATASET == "fashion" else 3
LR = 0.05
BATCH = 64
EST_SIZE = 2000
K = 20
BUF = (10, 50)
HIDDEN = 256
print("dataset:", DATASET); print("data:", DATA_DIR); print("results:", RESULTS_DIR)

dataset: cifar_feat
data: F:\forgetting\data
results: F:\forgetting\results_cifar_feat


## Cell 2: GPU + libraries
**Should print:** `cuda` and the 3090.

In [8]:
import time, json, csv
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.func import functional_call, vmap, grad as fgrad
from scipy.stats import spearmanr
from scipy.sparse.linalg import LinearOperator, eigsh
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", dev)
if dev.type == "cuda": print(torch.cuda.get_device_name(0))

torch 2.5.1+cu121 | device: cuda
NVIDIA GeForce RTX 3090


## Cell 3: load the data
- **`fashion`**: downloads Fashion-MNIST once (about 30 MB).
- **`cifar_feat`**: downloads ResNet-18 once (about 45 MB), then turns all 60,000 CIFAR photos into 512 numbers each. That takes a few minutes the first time; the result is saved, so later runs load it in seconds.

**Should print:** the input size (784 or 512) and 5 task lines.

In [9]:
from torchvision import datasets
if DATASET == "fashion":
    tr = datasets.FashionMNIST(DATA_DIR, train=True, download=True)
    te = datasets.FashionMNIST(DATA_DIR, train=False, download=True)
    prep = lambda ds: ((ds.data.float().div(255.).view(-1, 784) - 0.2860) / 0.3530, ds.targets.clone())
    Xtr_all, ytr_all = prep(tr); Xte_all, yte_all = prep(te)
    NAMES = ["T-shirt vs trouser", "pullover vs dress", "coat vs sandal", "shirt vs sneaker", "bag vs ankle boot"]
else:
    cache = os.path.join(DATA_DIR, "cifar10_resnet18_features.pt")
    if os.path.exists(cache):
        Xtr_all, ytr_all, Xte_all, yte_all = torch.load(cache)
        print("loaded saved features")
    else:
        from torchvision.models import resnet18, ResNet18_Weights
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1); net.fc = nn.Identity(); net = net.eval().to(dev)
        m = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); s = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
        def feats(ds):
            X = torch.tensor(ds.data).permute(0, 3, 1, 2); out = []
            for i in range(0, len(X), 250):
                xb = X[i:i + 250].to(dev).float().div(255.)
                xb = (F.interpolate(xb, size=224, mode="bilinear", align_corners=False) - m) / s
                with torch.no_grad(): out.append(net(xb).cpu())
            return torch.cat(out)
        tr = datasets.CIFAR10(DATA_DIR, train=True, download=True); te = datasets.CIFAR10(DATA_DIR, train=False, download=True)
        t0 = time.time(); Ftr = feats(tr); Fte = feats(te)
        mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
        Xtr_all, Xte_all = (Ftr - mu) / sd, (Fte - mu) / sd
        ytr_all, yte_all = torch.tensor(tr.targets), torch.tensor(te.targets)
        torch.save((Xtr_all, ytr_all, Xte_all, yte_all), cache)
        print(f"features made in {time.time() - t0:.0f}s and saved"); del net
    NAMES = ["airplane vs automobile", "bird vs cat", "deer vs dog", "frog vs horse", "ship vs truck"]
IN_DIM = Xtr_all.shape[1]
print("input size:", IN_DIM, "| train", len(ytr_all), "test", len(yte_all))
TASKS = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]
data = {}
g0 = torch.Generator().manual_seed(123)
for t, cls in enumerate(TASKS):
    mtr = (ytr_all == cls[0]) | (ytr_all == cls[1]); mte = (yte_all == cls[0]) | (yte_all == cls[1])
    Xtr, ytr, Xte, yte = Xtr_all[mtr], ytr_all[mtr], Xte_all[mte], yte_all[mte]
    est = torch.randperm(len(ytr), generator=g0)[:EST_SIZE]
    data[t] = dict(Xtr=Xtr.to(dev), ytr=ytr.to(dev), Xte=Xte.to(dev), yte=yte.to(dev),
                   Xest=Xtr[est].to(dev), yest=ytr[est].to(dev), cls=torch.tensor(cls, device=dev))
    print(f"task {t} ({NAMES[t]}): train {len(ytr)}, test {len(yte)}")

C:\Users\SignTalk\AppData\Local\Temp\ipykernel_22264\228613546.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  Xtr_all, ytr_all, Xte_all, yte_all = torch.load(cache)


loaded saved features
input size: 512 | train 50000 test 10000
task 0 (airplane vs automobile): train 10000, test 2000
task 1 (bird vs cat): train 10000, test 2000
task 2 (deer vs dog): train 10000, test 2000
task 3 (frog vs horse): train 10000, test 2000
task 4 (ship vs truck): train 10000, test 2000


## Cell 4: MLP and the maths
The same tanh MLP as on MNIST (two hidden layers of 256), plain SGD.

**Should print:** `ready` and the number of weights.

In [10]:
SCENARIO = "task"
class MLP(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.l1 = nn.Linear(IN_DIM, HIDDEN); self.l2 = nn.Linear(HIDDEN, HIDDEN); self.l3 = nn.Linear(HIDDEN, out_dim)
    def forward(self, x):
        return self.l3(torch.tanh(self.l2(torch.tanh(self.l1(x)))))

def setup(scenario):
    global SCENARIO, model, P_NAMES, P_SHAPES, P_SIZES, NPARAM
    SCENARIO = scenario
    model = MLP(2 if scenario == "domain" else 10).to(dev)
    P_NAMES = [n for n, _ in model.named_parameters()]
    P_SHAPES = [p.shape for _, p in model.named_parameters()]
    P_SIZES = [p.numel() for _, p in model.named_parameters()]
    NPARAM = sum(P_SIZES)

def unflat(th):
    out, i = {}, 0
    for n, s, k in zip(P_NAMES, P_SHAPES, P_SIZES):
        out[n] = th[i:i + k].view(s); i += k
    return out

def init(seed):
    torch.manual_seed(seed)
    m = MLP(2 if SCENARIO == "domain" else 10).to(dev)
    return torch.cat([p.detach().flatten() for p in m.parameters()])

def logits(th, X, cls):
    z = functional_call(model, unflat(th), (X,))
    return z if SCENARIO == "domain" else z[:, cls]

def task_loss(th, X, y, cls):
    return F.cross_entropy(logits(th, X, cls), (y == cls[1]).long())

def grad_on(th, X, y, cls):
    th = th.detach().requires_grad_(True)
    return torch.autograd.grad(task_loss(th, X, y, cls), th)[0].detach()

def hvp_on(th, X, y, cls, v):
    th = th.detach().requires_grad_(True)
    g = torch.autograd.grad(task_loss(th, X, y, cls), th, create_graph=True)[0]
    return torch.autograd.grad(g @ v.detach(), th)[0].detach()

def grad(th, t, X=None, y=None):
    d = data[t]
    return grad_on(th, d["Xest"] if X is None else X, d["yest"] if y is None else y, d["cls"])

def hvp(th, t, v):
    d = data[t]; return hvp_on(th, d["Xest"], d["yest"], d["cls"], v)

@torch.no_grad()
def loss_on(th, X, y, cls):
    return float(task_loss(th, X, y, cls))

@torch.no_grad()
def evaluate(th, t, split="te"):
    d = data[t]; X, y = d["X" + split], d["y" + split]
    z = logits(th, X, d["cls"]); yi = (y == d["cls"][1]).long()
    return float((z.argmax(1) == yi).float().mean()), float(F.cross_entropy(z, yi))

def train(th, t, seed, record=None, record_every=5):
    d = data[t]; th = th.detach().clone()
    gen = torch.Generator().manual_seed(seed * 1000 + t)
    n = len(d["ytr"]); step = 0
    for ep in range(EPOCHS):
        idx = torch.randperm(n, generator=gen).to(dev)
        for i in range(0, n, BATCH):
            bi = idx[i:i + BATCH]
            if record is not None and step % record_every == 0: record(step, th)
            th = th - LR * grad(th, t, d["Xtr"][bi], d["ytr"][bi])
            step += 1
    if record is not None: record(step, th)
    return th

def top_hessian(th, t, k=K):
    def mv(v):
        v_t = torch.as_tensor(np.asarray(v).ravel(), dtype=torch.float32, device=dev)
        return hvp(th, t, v_t).double().cpu().numpy()
    vals, vecs = eigsh(LinearOperator((NPARAM, NPARAM), matvec=mv, dtype=np.float64), k=k, which="LA", tol=1e-3)
    return torch.tensor(vals, dtype=torch.float32, device=dev), torch.tensor(vecs, dtype=torch.float32, device=dev)

def diag_fisher(th, t, chunk=100):
    d = data[t]; cls = d["cls"]
    per = vmap(fgrad(lambda th_, x, y: task_loss(th_, x[None], y[None], cls)), in_dims=(None, 0, 0))
    Fd = torch.zeros(NPARAM, device=dev)
    for i in range(0, len(d["yest"]), chunk):
        Fd += (per(th.detach(), d["Xest"][i:i + chunk], d["yest"][i:i + chunk]) ** 2).sum(0)
    return Fd / len(d["yest"])

def make_summary(th, t):
    lam, U = top_hessian(th, t)
    return dict(theta=th.detach().clone(), g=grad(th, t), lam=lam, U=U, F=diag_fisher(th, t))

def H_stored(S, v):
    return S["U"] @ (S["lam"] * (S["U"].T @ v))

setup("task"); print("ready | parameters:", NPARAM)

ready | parameters: 199690


## Cell 5: quick checks + is A well learned? (a few minutes)
1. **Symmetry:** two nearly equal numbers.
2. **Accuracy of every task, trained alone:** this is the key line. **All five should be above about 0.90.**
3. **Formula check:** ratio close to 1.00 at small learning rates.
4. **Stored map:** direction match at the switch.
5. **Valley shape of task 0:** the largest and the most negative curvature. A well-learned task should have hardly any negative curvature.

**Send Claude this output before running Cell 7.**

In [7]:
setup("domain")
th = init(99)
u, v = torch.randn(NPARAM, device=dev), torch.randn(NPARAM, device=dev)
print(f"1. symmetry: {float(u @ hvp(th, 0, v)):.5f}  vs  {float(v @ hvp(th, 0, u)):.5f}")
t0 = time.time(); accs_alone = [evaluate(train(init(0), t, 0), t)[0] for t in range(5)]
print("2. accuracy per task:", " | ".join(f"{NAMES[t]} {accs_alone[t]:.3f}" for t in range(5)), f"  (5 tasks took {time.time() - t0:.1f}s)")
thA = train(init(0), 0, 0)
th = init(0); ga, gb = grad(th, 0), grad(th, 1); br = hvp(th, 1, ga) - hvp(th, 0, gb)
for eta in [0.1, 0.03, 0.01, 0.003]:
    t1 = th - eta * ga; tAB = t1 - eta * grad(t1, 1)
    t2 = th - eta * gb; tBA = t2 - eta * grad(t2, 0)
    d = tAB - tBA; pr = eta ** 2 * br
    print(f"3. lr {eta:<6} ratio {float(d.norm() / pr.norm()):.3f}  direction match {float(F.cosine_similarity(d, pr, dim=0)):.3f}")
t0 = time.time(); S = make_summary(thA, 0)
gA, gB = grad(thA, 0), grad(thA, 1)
full = hvp(thA, 1, gA) - hvp(thA, 0, gB); st = hvp(thA, 1, S["g"]) - H_stored(S, gB)
print(f"4. summary built in {time.time() - t0:.1f}s | stored vs full at the switch: direction {float(F.cosine_similarity(full, st, dim=0)):.3f}, size {float(st.norm() / full.norm()):.3f}")
def mv(v):
    v_t = torch.as_tensor(np.asarray(v).ravel(), dtype=torch.float32, device=dev)
    return hvp(thA, 0, v_t).double().cpu().numpy()
op = LinearOperator((NPARAM, NPARAM), matvec=mv, dtype=np.float64)
lmax = eigsh(op, k=1, which="LA", tol=1e-3, return_eigenvectors=False).max()
lmin = eigsh(op, k=1, which="SA", tol=1e-3, return_eigenvectors=False).min()
print(f"5. task 0 valley: largest curvature {lmax:.4f}, most negative {lmin:.4f}, ratio {abs(min(lmin, 0)) / lmax:.3f}")

1. symmetry: 17.81909  vs  17.81911
2. accuracy per task: T-shirt vs trouser 0.986 | pullover vs dress 0.969 | coat vs sandal 1.000 | shirt vs sneaker 1.000 | bag vs ankle boot 0.997   (5 tasks took 6.7s)
3. lr 0.1    ratio 0.878  direction match 0.951
3. lr 0.03   ratio 0.979  direction match 0.995
3. lr 0.01   ratio 0.995  direction match 0.999
3. lr 0.003  ratio 0.999  direction match 1.000
4. summary built in 3.3s | stored vs full at the switch: direction 1.000, size 1.000
5. task 0 valley: largest curvature 9.3120, most negative -0.4041, ratio 0.043


## Cell 6: the experiment function (everything in one pass)
**Should print:** `defined`.

In [11]:
KEYS = ["bracket_stored", "ewc_penalty", "est_watch", "bracket_full", "simple_check"] + \
       [f"bracket_buf{n}" for n in BUF] + [f"check_buf{n}" for n in BUF]
LABELS = {"bracket_stored": "Bracket, stored map (no data)", "ewc_penalty": "EWC penalty (no data)",
          "est_watch": "Estimated watching (no data)", "bracket_buf10": "Bracket, 10 images", "check_buf10": "Simple check, 10 images",
          "bracket_buf50": "Bracket, 50 images", "check_buf50": "Simple check, 50 images",
          "bracket_full": "Ref: full bracket (old data)", "simple_check": "Ref: simple check (old data)"}

def run(scenario, lr=0.05, seeds=SEEDS):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    vals = {k: [] for k in KEYS}; target, forget, accs = [], [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t) for t in range(5)}
        base_est = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        summ = {t: make_summary(th_task[t], t) for t in range(5)}
        buf = {}
        for t in range(5):
            d = data[t]; g = torch.Generator().manual_seed(seed * 77 + t)
            idx = torch.randperm(len(d["ytr"]), generator=g)[:max(BUF)].to(dev)
            buf[t] = {n: (d["Xtr"][idx[:n]], d["ytr"][idx[:n]]) for n in BUF}
        accs += [base[t][0] for t in range(5)]
        for a, b in pairs:
            S = summ[a]; cA = data[a]["cls"]; log = []
            base_buf = {n: loss_on(th_task[a], *buf[a][n], cA) for n in BUF}
            def rec(step, th_, S=S, a=a, b=b, cA=cA, log=log, base_buf=base_buf):
                if step not in (5, 10, 15, 20): return
                dth = th_ - S["theta"]; gB = grad(th_, b)
                gA_hat = S["g"] + H_stored(S, dth); Ud = S["U"].T @ dth
                row = {"bracket_stored": float((hvp(th_, b, gA_hat) - H_stored(S, gB)).norm()),
                       "ewc_penalty": float((S["F"] * dth ** 2).sum()),
                       "est_watch": float(S["g"] @ dth + 0.5 * (S["lam"] * Ud ** 2).sum()),
                       "bracket_full": float((hvp(th_, b, grad(th_, a)) - hvp(th_, a, gB)).norm()),
                       "simple_check": evaluate(th_, a, "est")[1] - base_est[a]}
                for n in BUF:
                    Xb, yb = buf[a][n]
                    row[f"bracket_buf{n}"] = float((hvp(th_, b, grad_on(th_, Xb, yb, cA)) - hvp_on(th_, Xb, yb, cA, gB)).norm())
                    row[f"check_buf{n}"] = loss_on(th_, Xb, yb, cA) - base_buf[n]
                log.append(row)
            thAB = train(th_task[a], b, seed, record=rec)
            accA1, lossA1 = evaluate(thAB, a)
            target.append(lossA1 - base[a][1]); forget.append(base[a][0] - accA1)
            for k in KEYS: vals[k].append(float(np.mean([r[k] for r in log])))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s | mean task accuracy {np.mean([base[t][0] for t in range(5)]):.4f}")
    rho = {k: float(spearmanr(vals[k], target)[0]) for k in KEYS}
    Sm = {"scenario": scenario, "lr": lr, "seeds": seeds, "mean_task_accuracy": float(np.mean(accs)),
          "forgetting_mean": float(np.mean(forget)), "forgetting_max": float(np.max(forget)), "rho": rho}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"{DATASET}_{scenario}_lr{lr}.json"), "w"), indent=2)
    with open(os.path.join(RESULTS_DIR, f"{DATASET}_{scenario}_lr{lr}.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["loss_increase", "forgetting"] + KEYS)
        for i in range(len(target)): w.writerow([target[i], forget[i]] + [vals[k][i] for k in KEYS])
    print(f"\n== {DATASET} | {'slow' if scenario == 'task' else 'fast'} ({scenario}) | lr {lr} | task acc {Sm['mean_task_accuracy']:.3f} | "
          f"forgetting mean {Sm['forgetting_mean']:.3f}, max {Sm['forgetting_max']:.3f}")
    for k in ["bracket_stored", "ewc_penalty", "est_watch", "bracket_buf10", "check_buf10", "bracket_buf50", "check_buf50", "bracket_full", "simple_check"]:
        print(f"  {LABELS[k]:34s} {rho[k]:6.2f}")
    return Sm
print("defined")

defined


## Cell 7: main run, lr 0.05, both settings
Send Claude both tables.

In [9]:
RES = {}
for sc in ["task", "domain"]:
    RES[(sc, 0.05)] = run(sc, lr=0.05)

  seed 0 done after 67s | mean task accuracy 0.9901
  seed 1 done after 136s | mean task accuracy 0.9904
  seed 2 done after 202s | mean task accuracy 0.9905
  seed 3 done after 271s | mean task accuracy 0.9899
  seed 4 done after 341s | mean task accuracy 0.9906

== fashion | slow (task) | lr 0.05 | task acc 0.990 | forgetting mean 0.009, max 0.089
  Bracket, stored map (no data)        0.70
  EWC penalty (no data)                0.33
  Estimated watching (no data)         0.31
  Bracket, 10 images                   0.41
  Simple check, 10 images              0.29
  Bracket, 50 images                   0.52
  Simple check, 50 images              0.29
  Ref: full bracket (old data)         0.74
  Ref: simple check (old data)         0.33
  seed 0 done after 58s | mean task accuracy 0.9903
  seed 1 done after 120s | mean task accuracy 0.9907
  seed 2 done after 179s | mean task accuracy 0.9904
  seed 3 done after 237s | mean task accuracy 0.9899
  seed 4 done after 298s | mean task accu

## Cell 8: small steps (lr 0.02)
Optional if time is short.

In [ ]:
for sc in ["task", "domain"]:
    RES[(sc, 0.02)] = run(sc, lr=0.02)

In [9]:
import csv as _csv
for sc in ["task", "domain"]:
    f = os.path.join(RESULTS_DIR, f"{DATASET}_{sc}_lr0.05.csv")
    rows = list(_csv.DictReader(open(f)))
    y = np.array([float(r["loss_increase"]) for r in rows])
    rk = lambda k: np.argsort(np.argsort([float(r[k]) for r in rows]))   # rank of each run
    combos = {"bracket (no data)": rk("bracket_stored"), "EWC (no data)": rk("ewc_penalty"),
              "est. watching (no data)": rk("est_watch"),
              "bracket + EWC (no data)": rk("bracket_stored") + rk("ewc_penalty"),
              "bracket + EWC + est. watching": rk("bracket_stored") + rk("ewc_penalty") + rk("est_watch"),
              "bracket 50 imgs + EWC": rk("bracket_buf50") + rk("ewc_penalty")}
    print(f"\n{'slow' if sc == 'task' else 'fast'}")
    for name, v in combos.items():
        print(f"  {name:32s} {spearmanr(v, y)[0]:6.2f}")


slow
  bracket (no data)                  0.76
  EWC (no data)                      0.35
  est. watching (no data)            0.58
  bracket + EWC (no data)            0.62
  bracket + EWC + est. watching      0.69
  bracket 50 imgs + EWC              0.57

fast
  bracket (no data)                  0.46
  EWC (no data)                      0.84
  est. watching (no data)            0.71
  bracket + EWC (no data)            0.71
  bracket + EWC + est. watching      0.75
  bracket 50 imgs + EWC              0.81


## Cell 9: order test + the slope/curvature test (same cells as yesterday and this morning)
Prints two tables per setting. Send Claude everything.

In [10]:
def run_order(scenario, lr=0.05, seeds=3):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    K_ = ["br_start", "br_onA_start", "br_early", "forgetting_loss"]
    vals = {k: [] for k in K_}; gapA, gapP = [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base = {t: evaluate(th_task[t], t)[1] for t in range(5)}
        g0 = {t: grad(th0, t) for t in range(5)}
        thAB, early = {}, {}
        for a, b in pairs:
            log = []
            def rec(step, th_, a=a, b=b, log=log):
                if step in (5, 10, 15, 20):
                    log.append(float((hvp(th_, b, grad(th_, a)) - hvp(th_, a, grad(th_, b))).norm()))
            thAB[(a, b)] = train(th_task[a], b, seed, record=rec)
            early[(a, b)] = float(np.mean(log))
        for a, b in pairs:
            br0 = hvp(th0, b, g0[a]) - hvp(th0, a, g0[b])
            lossA_AB = evaluate(thAB[(a, b)], a)[1]          # A learned first, then B
            lossA_BA = evaluate(thAB[(b, a)], a)[1]          # A learned last
            gapA.append(lossA_AB - lossA_BA)
            gapP.append(float((thAB[(a, b)] - thAB[(b, a)]).norm()))
            vals["br_start"].append(float(br0.norm()))
            vals["br_onA_start"].append(float(g0[a] @ br0))
            vals["br_early"].append(early[(a, b)])
            vals["forgetting_loss"].append(lossA_AB - base[a])
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    r = lambda x, y: float(spearmanr(x, y)[0])
    out = {
        "order_gap_on_A vs br_start": r(vals["br_start"], gapA),
        "order_gap_on_A vs br_onA_start": r(vals["br_onA_start"], gapA),
        "order_gap_on_A vs br_early": r(vals["br_early"], gapA),
        "weight_gap vs br_start": r(vals["br_start"], gapP),
        "weight_gap vs br_early": r(vals["br_early"], gapP),
        "forgetting vs order_gap_on_A": r(gapA, vals["forgetting_loss"]),
        "forgetting vs br_early": r(vals["br_early"], vals["forgetting_loss"]),
    }
    share = float(np.mean(np.abs(gapA)) / np.mean(np.abs(vals["forgetting_loss"])))
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds, "rho": out, "order_share_of_forgetting": share}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"order_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== ORDER | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in out.items(): print(f"  {k:34s} {v:6.2f}")
    print(f"  order effect as share of forgetting  {share:6.2f}")
    return Sm



def run_mech(scenario, lr=0.05, seeds=3):
    global LR
    LR = lr; setup(scenario)
    pairs = [(a, b) for a in range(5) for b in range(5) if a != b]
    R = {k: [] for k in ["forget", "push_ratio", "gA_norm", "term_HBgA", "term_HAgB", "bracket",
                         "slope20", "curv20", "actual20", "slope_end", "curv_end", "actual_end"]}
    t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        base_te = {t: evaluate(th_task[t], t)[1] for t in range(5)}
        base_est = {t: evaluate(th_task[t], t, "est")[1] for t in range(5)}
        gA0 = {t: grad(th_task[t], t) for t in range(5)}          # A's push when A finishes
        for a, b in pairs:
            thA = th_task[a]; log = []; at20 = {}
            def rec(step, th_, a=a, b=b, thA=thA, log=log, at20=at20):
                if step not in (5, 10, 15, 20): return
                gA, gB = grad(th_, a), grad(th_, b)
                t1, t2 = hvp(th_, b, gA), hvp(th_, a, gB)
                log.append((float(t1.norm()), float(t2.norm()), float((t1 - t2).norm())))
                if step == 20: at20["th"] = th_.detach().clone()
            thAB = train(thA, b, seed, record=rec)
            def split(th_):
                d = th_ - thA
                slope = float(gA0[a] @ d)
                curv = float(0.5 * d @ hvp(thA, a, d))
                actual = evaluate(th_, a, "est")[1] - base_est[a]
                return slope, curv, actual
            s20, c20, a20 = split(at20["th"]); se, ce, ae = split(thAB)
            R["forget"].append(evaluate(thAB, a)[1] - base_te[a])
            R["gA_norm"].append(float(gA0[a].norm()))
            R["push_ratio"].append(float(gA0[a].norm() / grad(thA, b).norm()))
            R["term_HBgA"].append(float(np.mean([x[0] for x in log])))
            R["term_HAgB"].append(float(np.mean([x[1] for x in log])))
            R["bracket"].append(float(np.mean([x[2] for x in log])))
            for k, v in zip(["slope20", "curv20", "actual20", "slope_end", "curv_end", "actual_end"], [s20, c20, a20, se, ce, ae]):
                R[k].append(v)
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    r = lambda k: float(spearmanr(R[k], R["forget"])[0])
    share = lambda s, c: float(np.mean(np.abs(R[s])) / (np.mean(np.abs(R[s])) + np.mean(np.abs(R[c]))))
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds,
          "c1_push_A_mean": float(np.mean(R["gA_norm"])), "c1_push_ratio_A_over_B": float(np.mean(R["push_ratio"])),
          "c2_slope_share_step20": share("slope20", "curv20"), "c2_slope_share_end": share("slope_end", "curv_end"),
          "c2_taylor_vs_actual_step20": float(spearmanr(np.array(R["slope20"]) + np.array(R["curv20"]), R["actual20"])[0]),
          "c3_rho_full_bracket": r("bracket"), "c3_rho_HBgA": r("term_HBgA"), "c3_rho_HAgB": r("term_HAgB"),
          "c3_rho_slope20": r("slope20"), "c3_rho_curv20": r("curv20")}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"mech_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== MECH | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in Sm.items():
        if k.startswith("c"): print(f"  {k:30s} {v:8.3f}")
    return Sm


for sc in ["task", "domain"]:
    run_order(sc, lr=0.05)
    run_mech(sc, lr=0.05)

  seed 0 done after 44s
  seed 1 done after 88s
  seed 2 done after 132s

== ORDER | fashion | slow | lr 0.05
  order_gap_on_A vs br_start           0.14
  order_gap_on_A vs br_onA_start      -0.22
  order_gap_on_A vs br_early           0.68
  weight_gap vs br_start               0.51
  weight_gap vs br_early              -0.06
  forgetting vs order_gap_on_A         0.99
  forgetting vs br_early               0.67
  order effect as share of forgetting    1.00
  seed 0 done after 44s
  seed 1 done after 87s
  seed 2 done after 131s

== MECH | fashion | slow | lr 0.05
  c1_push_A_mean                    0.091
  c1_push_ratio_A_over_B            0.036
  c2_slope_share_step20             0.349
  c2_slope_share_end                0.241
  c2_taylor_vs_actual_step20        0.941
  c3_rho_full_bracket               0.672
  c3_rho_HBgA                       0.495
  c3_rho_HAgB                       0.711
  c3_rho_slope20                   -0.295
  c3_rho_curv20                     0.710
  seed 

## Cell 10: valley shape (is A sitting at a true bottom?)
For each task: the largest and the most negative curvature, and how much B's push lines up with A's curvature
(`cos_gB_HAgB`: near 1 = clean bowl, near 0 or negative = saddle-like, where the bracket's size measure misleads).

In [11]:
from scipy.sparse.linalg import LinearOperator, eigsh
if "DS" not in globals(): DS = "cifar" if "cifar" in RESULTS_DIR else "mnist"

def curv_shape(scenario, lr=0.05, seeds=2):
    global LR
    LR = lr; setup(scenario)
    lmax, lmin, cosv = [], [], []; t0 = time.time()
    for seed in range(seeds):
        th0 = init(seed)
        th_task = {t: train(th0, t, seed) for t in range(5)}
        for a in range(5):
            th = th_task[a]
            def mv(v, th=th, a=a):
                v_t = torch.as_tensor(np.asarray(v).ravel(), dtype=torch.float32, device=dev)
                return hvp(th, a, v_t).double().cpu().numpy()
            op = LinearOperator((NPARAM, NPARAM), matvec=mv, dtype=np.float64)
            lmax.append(float(eigsh(op, k=1, which="LA", tol=1e-3, return_eigenvectors=False).max()))
            lmin.append(float(eigsh(op, k=1, which="SA", tol=1e-3, return_eigenvectors=False).min()))
            for b in range(5):
                if b != a:
                    gB = grad(th, b); cosv.append(float(F.cosine_similarity(gB, hvp(th, a, gB), dim=0)))
        print(f"  seed {seed} done after {time.time() - t0:.0f}s")
    ratio = [abs(min(mn, 0)) / mx for mn, mx in zip(lmin, lmax)]
    Sm = {"dataset": DS, "scenario": scenario, "lr": lr, "seeds": seeds,
          "largest_curvature_mean": float(np.mean(lmax)), "most_negative_mean": float(np.mean(lmin)),
          "neg_to_pos_ratio": float(np.mean(ratio)), "cos_gB_HAgB_mean": float(np.mean(cosv)),
          "cos_below_0.3_share": float(np.mean(np.array(cosv) < 0.3))}
    json.dump(Sm, open(os.path.join(RESULTS_DIR, f"curv_{DS}_{scenario}_lr{lr}.json"), "w"), indent=2)
    print(f"\n== VALLEY | {DS} | {'slow' if scenario == 'task' else 'fast'} | lr {lr}")
    for k, v in Sm.items():
        if k not in ("dataset", "scenario", "lr", "seeds"): print(f"  {k:24s} {v:9.4f}")
    return Sm

for sc in ["task", "domain"]:
    curv_shape(sc, lr=0.05)

  seed 0 done after 15s
  seed 1 done after 28s

== VALLEY | fashion | slow | lr 0.05
  largest_curvature_mean      3.8804
  most_negative_mean         -0.2037
  neg_to_pos_ratio            0.0614
  cos_gB_HAgB_mean            0.0772
  cos_below_0.3_share         1.0000
  seed 0 done after 11s
  seed 1 done after 22s

== VALLEY | fashion | fast | lr 0.05
  largest_curvature_mean      3.8500
  most_negative_mean         -0.2295
  neg_to_pos_ratio            0.0715
  cos_gB_HAgB_mean            0.6235
  cos_below_0.3_share         0.0500


## Cell 11: summary table + headline chart (reads from the drive)

In [ ]:
import glob
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Calibri", "font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
R = {}
for f in glob.glob(os.path.join(RESULTS_DIR, f"{DATASET}_*.json")):
    d = json.load(open(f)); R[(d["scenario"], d["lr"])] = d
LRS = sorted({lr for _, lr in R})
cols = ["ewc_penalty", "bracket_stored", "bracket_buf10", "check_buf10", "bracket_buf50", "check_buf50", "bracket_full", "simple_check"]
heads = ["EWC", "Br stored", "Br 10", "Chk 10", "Br 50", "Chk 50", "Ref full br", "Ref check"]
print(f"{'setting':16s}" + "".join(f"{h:>12s}" for h in heads))
for sc in ["task", "domain"]:
    for lr in LRS:
        if (sc, lr) in R:
            r = R[(sc, lr)]["rho"]
            print(f"{('slow' if sc == 'task' else 'fast') + ' lr ' + str(lr):16s}" + "".join(f"{r[c]:12.2f}" for c in cols))
labels, br, ew = [], [], []
for sc, key, tag in [("task", "bracket_stored", "Slow, no data"), ("domain", "bracket_buf10", "Fast, 10 images")]:
    for lr in LRS:
        if (sc, lr) in R:
            labels.append(f"{tag}\nlr {lr}"); br.append(R[(sc, lr)]["rho"][key]); ew.append(R[(sc, lr)]["rho"]["ewc_penalty"])
fig, ax = plt.subplots(figsize=(9, 4.6), dpi=200); x = np.arange(len(labels)); w = 0.38
ax.bar(x - w / 2, br, w, color="#1F8A70", label="Lie bracket"); ax.bar(x + w / 2, ew, w, color="#B8651C", label="EWC (no data)")
for xi, v in zip(x - w / 2, br): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=10)
for xi, v in zip(x + w / 2, ew): ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10); ax.set_ylim(min(0, min(br + ew) - 0.05), 1)
ax.set_ylabel("Rank correlation with forgetting"); ax.grid(axis="y", color="#E3E8E6"); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper right")
ax.set_title(f"{DATASET}: Lie bracket vs EWC at predicting forgetting", loc="left", weight="bold", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, f"{DATASET}_headline.png"), facecolor="white"); plt.close(fig)
print("saved", os.path.join(RESULTS_DIR, f"{DATASET}_headline.png"))

In [6]:
import csv as _csv
rows = list(_csv.DictReader(open(os.path.join(RESULTS_DIR, f"{DATASET}_domain_lr0.05.csv"))))
y = np.array([float(r["loss_increase"]) for r in rows])
rk = lambda k: np.argsort(np.argsort([float(r[k]) for r in rows]))
for name, v in {"bracket": rk("bracket_stored"), "est. watching": rk("est_watch"), "EWC": rk("ewc_penalty"),
                "bracket + est. watching": rk("bracket_stored") + rk("est_watch")}.items():
    print(f"  {name:26s} {spearmanr(v, y)[0]:6.2f}")

  bracket                      0.46
  est. watching                0.71
  EWC                          0.84
  bracket + est. watching      0.62


In [12]:
def make_summary(th, t):                       # redefine so the map size K is used
    lam, U = top_hessian(th, t, k=K)
    return dict(theta=th.detach().clone(), g=grad(th, t), lam=lam, U=U, F=diag_fisher(th, t))

for K in [50, 100]:
    RESULTS_DIR = os.path.join(BASE, f"results_{DATASET}_K{K}"); os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"\n######## stored map with {K} directions ########")
    run("domain", lr=0.05, seeds=3)
RESULTS_DIR = os.path.join(BASE, "results_" + DATASET); K = 20


######## stored map with 50 directions ########
  seed 0 done after 80s | mean task accuracy 0.9716
  seed 1 done after 155s | mean task accuracy 0.9736
  seed 2 done after 232s | mean task accuracy 0.9731

== cifar_feat | fast (domain) | lr 0.05 | task acc 0.973 | forgetting mean 0.185, max 0.462
  Bracket, stored map (no data)        0.49
  EWC penalty (no data)                0.85
  Estimated watching (no data)         0.76
  Bracket, 10 images                   0.17
  Simple check, 10 images              0.25
  Bracket, 50 images                   0.63
  Simple check, 50 images              0.62
  Ref: full bracket (old data)         0.80
  Ref: simple check (old data)         0.84

######## stored map with 100 directions ########
  seed 0 done after 193s | mean task accuracy 0.9716
  seed 1 done after 390s | mean task accuracy 0.9736
  seed 2 done after 597s | mean task accuracy 0.9731

== cifar_feat | fast (domain) | lr 0.05 | task acc 0.973 | forgetting mean 0.185, max 0.462
  